# Federated Learning: A Complete Learning Notebook

## Table of Contents
1. [Introduction](#introduction)
2. [Core Concepts](#core-concepts)
3. [How Federated Learning Works](#how-it-works)
4. [Federated Averaging Algorithm](#fedavg)
5. [Practical Implementation](#implementation)
6. [Challenges and Solutions](#challenges)
7. [Real-World Applications](#applications)
8. [Hands-On Exercises](#exercises)
9. [Further Reading](#further-reading)

---

## 1. Introduction

### What is Federated Learning?

Federated Learning (FL) is a machine learning approach that enables model training across multiple decentralized devices or servers holding local data samples, **without exchanging the raw data itself**.

### Key Motivation

Traditional ML requires centralizing data in one location:
```
Device 1 data  ──┐
Device 2 data  ──┼──> Central Server ──> Train Model
Device 3 data  ──┘
```

Problems with this approach:
- **Privacy concerns**: Sensitive data must leave devices
- **Bandwidth costs**: Transferring large datasets is expensive
- **Regulatory compliance**: GDPR, HIPAA restrict data movement
- **Data silos**: Organizations can't share proprietary data

Federated Learning solution:
```
Device 1 ──> Local training ──┐
Device 2 ──> Local training ──┼──> Aggregate updates ──> Global Model
Device 3 ──> Local training ──┘
```

### Historical Context

- **2016**: Google introduces Federated Learning for Gboard (mobile keyboard)
- **2017**: Federated Averaging (FedAvg) algorithm published
- **2019+**: Explosion in research, applications in healthcare, finance, IoT

---

## 2. Core Concepts

### 2.1 The Federated Learning Paradigm

**Three key principles:**

1. **Data remains local**: Raw data never leaves the device/organization
2. **Collaborative learning**: Multiple parties train a shared model
3. **Privacy preservation**: Only model updates (gradients/weights) are shared

### 2.2 Architecture Components

```
┌─────────────────────────────────────────────────────────────┐
│                    CENTRAL SERVER                           │
│  - Initializes global model                                 │
│  - Aggregates client updates                                │
│  - Distributes updated model                                │
└──────────────────┬──────────────────────────────────────────┘
                   │
        ┌──────────┼──────────┬──────────┐
        │          │          │          │
    ┌───▼───┐  ┌───▼───┐  ┌───▼───┐  ┌───▼───┐
    │Client1│  │Client2│  │Client3│  │ClientN│
    │Data D1│  │Data D2│  │Data D3│  │Data DN│
    └───────┘  └───────┘  └───────┘  └───────┘
```

### 2.3 Types of Federated Learning

| Type | Description | Example |
|------|-------------|---------|
| **Horizontal Federated Learning (HFL)** | Same feature space, different samples/users | Multiple hospitals with similar patient attributes (e.g., age, blood pressure, diagnosis) but different patients |
| **Vertical Federated Learning (VFL)** | Different feature spaces, same samples/users | A bank and an e-commerce company sharing the same customers but with different data (financial data vs. shopping behavior) |
| **Federated Transfer Learning (FTL)** | Different feature spaces and different samples/users, with limited overlap | Collaboration between organizations from different domains with minimal shared users and heterogeneous data |

### 2.4 Cross-Silo vs Cross-Device

**Cross-Silo FL:**
- Few participants (2-100 organizations)
- Reliable connections
- Larger datasets per participant
- Example: Hospital consortium

**Cross-Device FL:**
- Millions of participants (mobile devices)
- Unreliable connections
- Small datasets per device
- Example: Smartphone keyboards

---

## 3. How Federated Learning Works
### The Basic Workflow

```
Round t = 1, 2, 3, ...

1. SERVER INITIALIZATION
   ├─> Initialize global model w(t)
   └─> Select subset of clients C(t)

2. CLIENT SELECTION & DISTRIBUTION
   ├─> Send w(t) to selected clients
   └─> Each client k receives model

3. LOCAL TRAINING (at each client k)
   ├─> Train on local data D(k)
   ├─> Compute local update Δw(k)
   └─> Send update to server

4. AGGREGATION (at server)
   ├─> Collect updates from clients
   ├─> Aggregate: w(t+1) = w(t) + Σ(Δw(k))
   └─> Update global model

5. REPEAT
   └─> Go to step 1 with new round t+1
```

In [8]:
# Step-by-Step Example (Pseudocode)

# Setup: 3 hospitals want to train a disease prediction model
# Each has 1000 patient records
# Privacy regulations prevent data sharing

# Round 1:

# Server initializes model
# global_model = NeuralNetwork()
# global_weights = global_model.get_weights()

# Distribute to hospitals
# for hospital in [H1, H2, H3]:
#     hospital.receive_model(global_weights)

# Each hospital trains locally
# H1_update = H1.train(local_data=H1.patients, epochs=5)
# H2_update = H2.train(local_data=H2.patients, epochs=5)
# H3_update = H3.train(local_data=H3.patients, epochs=5)

# Server aggregates (simple average)
# new_weights = (H1_update + H2_update + H3_update) / 3
# global_model.set_weights(new_weights)

# Round 2-N: Repeat until convergence

print("This is a conceptual example - see implementation section for runnable code")

This is a conceptual example - see implementation section for runnable code


---

## 4. Federated Averaging Algorithm

### The FedAvg Algorithm

**Most widely used FL algorithm**, proposed by McMahan et al. (2017).

#### Mathematical Formulation

**Objective:** Minimize global loss function

```
min_w F(w) where F(w) = Σ(k=1 to K) (n_k/n) * F_k(w)
```

Where:
- `w` = global model parameters
- `K` = total number of clients
- `n_k` = number of samples at client k
- `n` = total samples across all clients
- `F_k(w)` = local loss function at client k

#### Algorithm Pseudocode

```
Server:
  Initialize w_0
  
  for each round t = 1, 2, 3, ..., T:
    m ← max(C · K, 1)  # Select C fraction of clients
    S_t ← random set of m clients
    
    for each client k ∈ S_t in parallel:
      w_t+1^k ← ClientUpdate(k, w_t)
    
    # Weighted aggregation
    w_t+1 ← Σ(k∈S_t) (n_k/n) * w_t+1^k

ClientUpdate(k, w):  # Run on client k
  B ← split local data into batches
  
  for i = 1 to E (local epochs):
    for batch b ∈ B:
      w ← w - η∇ℓ(w; b)  # SGD update
  
  return w to server
```

**Key parameters:**
- `C`: Fraction of clients selected per round (e.g., 0.1 = 10%)
- `E`: Number of local epochs
- `B`: Local batch size
- `η`: Learning rate

### Why FedAvg Works

1. **Local computation**: Reduces communication rounds
2. **Partial client participation**: Scalable to millions of devices
3. **Weighted averaging**: Accounts for data imbalance
4. **Proven convergence**: Under certain assumptions (convexity, IID data)

---

## 5. Practical Implementation

### 5.1 Simple Federated Learning Example (PyTorch)

Let's implement a complete federated learning system from scratch!

In [9]:
# Install required packages
!pip install torch numpy
!pip install flwr

In [10]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import copy
import numpy as np

print(f"PyTorch version: {torch.__version__}")
print(f"NumPy version: {np.__version__}")

PyTorch version: 2.10.0+cpu
NumPy version: 2.0.2


In [11]:
# Simple Neural Network
class SimpleNN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Test the model
test_model = SimpleNN(input_dim=10, hidden_dim=20, output_dim=2)
test_input = torch.randn(5, 10)
test_output = test_model(test_input)
print(f"Model output shape: {test_output.shape}")

Model output shape: torch.Size([5, 2])


In [12]:
# Client class
class FederatedClient:
    def __init__(self, client_id, data, labels):
        self.client_id = client_id
        self.data = data
        self.labels = labels
        self.model = None

    def receive_model(self, global_model):
        """Receive global model from server"""
        self.model = copy.deepcopy(global_model)

    def local_train(self, epochs=5, batch_size=32, lr=0.01):
        """Train model on local data"""
        dataset = TensorDataset(
            torch.FloatTensor(self.data),
            torch.LongTensor(self.labels)
        )
        dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

        criterion = nn.CrossEntropyLoss()
        optimizer = optim.SGD(self.model.parameters(), lr=lr)

        self.model.train()
        for epoch in range(epochs):
            epoch_loss = 0.0
            for batch_data, batch_labels in dataloader:
                optimizer.zero_grad()
                outputs = self.model(batch_data)
                loss = criterion(outputs, batch_labels)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()

            if epoch == epochs - 1:
                print(f"  Client {self.client_id} - Final epoch loss: {epoch_loss/len(dataloader):.4f}")

        return self.model.state_dict()

print("FederatedClient class defined successfully")

FederatedClient class defined successfully


In [13]:
# Server class
class FederatedServer:
    def __init__(self, model):
        self.global_model = model
        self.clients = []

    def add_client(self, client):
        """Register a client"""
        self.clients.append(client)

    def select_clients(self, num_clients):
        """Randomly select clients for this round"""
        import random
        return random.sample(self.clients, min(num_clients, len(self.clients)))

    def aggregate(self, client_models, client_weights):
        """FedAvg aggregation"""
        global_dict = self.global_model.state_dict()

        for key in global_dict.keys():
            global_dict[key] = torch.zeros_like(global_dict[key], dtype=torch.float32)

            for client_model, weight in zip(client_models, client_weights):
                global_dict[key] += client_model[key].float() * weight

        self.global_model.load_state_dict(global_dict)

    def train_round(self, num_clients, local_epochs=5):
        """Execute one round of federated learning"""
        # Select clients
        selected_clients = self.select_clients(num_clients)
        print(f"Selected clients: {[c.client_id for c in selected_clients]}")

        # Distribute model
        for client in selected_clients:
            client.receive_model(self.global_model)

        # Local training
        client_models = []
        client_weights = []
        total_samples = sum([len(c.data) for c in selected_clients])

        for client in selected_clients:
            print(f"Training on client {client.client_id}...")
            model_update = client.local_train(epochs=local_epochs)
            client_models.append(model_update)

            # Weight by data size
            weight = len(client.data) / total_samples
            client_weights.append(weight)

        # Aggregate
        self.aggregate(client_models, client_weights)
        print("Aggregation complete\n")

print("FederatedServer class defined successfully")

FederatedServer class defined successfully


In [14]:
# Simulation: Federated learning with 5 clients

def federated_learning_simulation():
    """Simulate federated learning with 5 clients"""

    print("Creating synthetic data for 5 clients (non-IID)...\n")

    # Create synthetic data for 5 clients (non-IID)
    clients_data = []
    for i in range(5):
        # Each client has different data distribution
        X = np.random.randn(200, 10) + i  # Different mean per client
        y = np.random.randint(0, 2, 200)
        clients_data.append((X, y))
        print(f"Client {i}: {len(X)} samples, mean feature value: {X.mean():.2f}")

    print("\nInitializing server and clients...\n")

    # Initialize server
    global_model = SimpleNN(input_dim=10, hidden_dim=20, output_dim=2)
    server = FederatedServer(global_model)

    # Create clients
    for i, (X, y) in enumerate(clients_data):
        client = FederatedClient(client_id=i, data=X, labels=y)
        server.add_client(client)

    # Training loop
    num_rounds = 5
    clients_per_round = 3

    print(f"Starting Federated Learning:")
    print(f"- Total rounds: {num_rounds}")
    print(f"- Clients per round: {clients_per_round}")
    print(f"- Local epochs: 5\n")
    print("="*60)

    for round_num in range(num_rounds):
        print(f"\n=== Round {round_num + 1}/{num_rounds} ===")
        server.train_round(num_clients=clients_per_round, local_epochs=5)

    print("\n" + "="*60)
    print("Federated Learning Complete!")
    print("="*60)
    return server.global_model

# Run simulation
final_model = federated_learning_simulation()

Creating synthetic data for 5 clients (non-IID)...

Client 0: 200 samples, mean feature value: 0.01
Client 1: 200 samples, mean feature value: 1.02
Client 2: 200 samples, mean feature value: 1.99
Client 3: 200 samples, mean feature value: 2.95
Client 4: 200 samples, mean feature value: 3.95

Initializing server and clients...

Starting Federated Learning:
- Total rounds: 5
- Clients per round: 3
- Local epochs: 5


=== Round 1/5 ===
Selected clients: [3, 1, 4]
Training on client 3...
  Client 3 - Final epoch loss: 0.6809
Training on client 1...
  Client 1 - Final epoch loss: 0.7121
Training on client 4...
  Client 4 - Final epoch loss: 0.6996
Aggregation complete


=== Round 2/5 ===
Selected clients: [4, 1, 0]
Training on client 4...
  Client 4 - Final epoch loss: 0.6843
Training on client 1...
  Client 1 - Final epoch loss: 0.7106
Training on client 0...
  Client 0 - Final epoch loss: 0.6934
Aggregation complete


=== Round 3/5 ===
Selected clients: [2, 0, 4]
Training on client 2...
 

In [15]:
# Test the final model
print("\nTesting final global model...")
test_data = torch.randn(10, 10)
final_model.eval()
with torch.no_grad():
    predictions = final_model(test_data)
    predicted_classes = predictions.argmax(dim=1)

print(f"Test input shape: {test_data.shape}")
print(f"Predictions: {predicted_classes.numpy()}")
print("\nModel trained successfully!")


Testing final global model...
Test input shape: torch.Size([10, 10])
Predictions: [0 0 1 0 0 0 0 0 0 0]

Model trained successfully!


### 5.2 Using Flower Framework

[Flower](https://flower.dev/) is a popular production-ready FL framework. This section demonstrates a complete Federated Learning (FL) workflow using the Flower framework and PyTorch. We simulate a distributed learning environment where multiple clients collaboratively train a shared global model on the MNIST dataset without sharing their raw data. This section covers the creation of a Flower server, multiple federated clients, local training, model aggregation using the FedAvg algorithm, and evaluation across several communication rounds.

##Server file

In [17]:
%%writefile server.py
import flwr as fl

strategy = fl.server.strategy.FedAvg(
    min_fit_clients=3,
    min_available_clients=3,
    min_evaluate_clients=3,
)

fl.server.start_server(
    server_address="0.0.0.0:9090",
    config=fl.server.ServerConfig(num_rounds=3),
    strategy=strategy,
)

Overwriting server.py


##Client file

In [19]:
%%writefile client.py
import sys
import flwr as fl
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

CLIENT_ID = int(sys.argv[1])
NUM_CLIENTS = 3

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 784)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)

def load_data(client_id):
    transform = transforms.ToTensor()

    trainset = datasets.MNIST("./data", train=True, download=True, transform=transform)
    testset = datasets.MNIST("./data", train=False, download=True, transform=transform)

    train_indices = list(range(client_id, len(trainset), NUM_CLIENTS))
    test_indices = list(range(client_id, len(testset), NUM_CLIENTS))

    trainloader = DataLoader(
        Subset(trainset, train_indices),
        batch_size=32,
        shuffle=True
    )

    testloader = DataLoader(
        Subset(testset, test_indices),
        batch_size=32,
        shuffle=False
    )

    return trainloader, testloader

class FlowerClient(fl.client.NumPyClient):
    def __init__(self, model, trainloader, testloader):
        self.model = model
        self.trainloader = trainloader
        self.testloader = testloader

    def get_parameters(self, config):
        print(f"Client {CLIENT_ID}: sending parameters", flush=True)
        return [val.cpu().numpy() for _, val in self.model.state_dict().items()]

    def set_parameters(self, parameters):
        params_dict = zip(self.model.state_dict().keys(), parameters)
        state_dict = {k: torch.tensor(v) for k, v in params_dict}
        self.model.load_state_dict(state_dict, strict=True)

    def fit(self, parameters, config):
        print(f"Client {CLIENT_ID}: training started", flush=True)

        self.set_parameters(parameters)

        optimizer = torch.optim.SGD(self.model.parameters(), lr=0.01)
        criterion = nn.CrossEntropyLoss()

        self.model.train()

        for epoch in range(1):
            for data, target in self.trainloader:
                optimizer.zero_grad()
                output = self.model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

        print(f"Client {CLIENT_ID}: training finished", flush=True)

        return self.get_parameters(config={}), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        print(f"Client {CLIENT_ID}: evaluation started", flush=True)

        self.set_parameters(parameters)

        self.model.eval()
        criterion = nn.CrossEntropyLoss()

        loss = 0.0
        correct = 0

        with torch.no_grad():
            for data, target in self.testloader:
                output = self.model(data)
                loss += criterion(output, target).item()
                pred = output.argmax(dim=1)
                correct += pred.eq(target).sum().item()

        accuracy = correct / len(self.testloader.dataset)

        print(f"Client {CLIENT_ID}: accuracy = {accuracy:.4f}", flush=True)

        return loss, len(self.testloader.dataset), {"accuracy": accuracy}

model = Net()
trainloader, testloader = load_data(CLIENT_ID)

fl.client.start_client(
    server_address="127.0.0.1:9090",
    client=FlowerClient(model, trainloader, testloader).to_client()
)

Overwriting client.py


##Stop old processes if they exist

In [20]:
for name in ["server_process", "client1", "client2", "client3"]:
    if name in globals():
        try:
            globals()[name].terminate()
            print(f"{name} stopped")
        except Exception:
            pass

##Start server with log file

In [21]:
import subprocess
import time

server_log = open("server.log", "w")

server_process = subprocess.Popen(
    ["python", "server.py"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    text=True
)

time.sleep(5)

print("Server started")

Server started


##Start 3 clients with log files

In [22]:
client_logs = []

client1_log = open("client_0.log", "w")
client2_log = open("client_1.log", "w")
client3_log = open("client_2.log", "w")

client1 = subprocess.Popen(
    ["python", "client.py", "0"],
    stdout=client1_log,
    stderr=subprocess.STDOUT,
    text=True
)

client2 = subprocess.Popen(
    ["python", "client.py", "1"],
    stdout=client2_log,
    stderr=subprocess.STDOUT,
    text=True
)

client3 = subprocess.Popen(
    ["python", "client.py", "2"],
    stdout=client3_log,
    stderr=subprocess.STDOUT,
    text=True
)

print("3 clients started")

3 clients started


##Wait a bit, then check logs

In [23]:
import time

time.sleep(20)

print("===== SERVER LOG =====")
with open("server.log", "r") as f:
    print(f.read())

for i in range(3):
    print(f"\n===== CLIENT {i} LOG =====")
    with open(f"client_{i}.log", "r") as f:
        print(f.read())

===== SERVER LOG =====
	Instead, use the `flower-superlink` CLI command to start a SuperLink as shown below:

		$ flower-superlink --insecure

	To view usage and all available options, run:

		$ flower-superlink --help

	Using `start_server()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
INFO :      Starting Flower server, config: num_rounds=3, no round_timeout
INFO :      Flower ECE: gRPC server running (3 rounds), SSL is disabled
INFO :      [INIT]
INFO :      Requesting initial parameters from one random client
INFO :      Received initial parameters from one random client
INFO :      Starting evaluation of initial global parameters
INFO :      Evaluation returned no results (`None`)
INFO :      
INFO :      [ROUND 1]
INFO :      configure_fit: strategy sampled 3 clients (out of 3)
INFO :      aggregate_fit: received 3 results and 0 failures
INFO :      configure_evaluate: strategy sampled 3 

##Stop everything

In [24]:
server_process.terminate()
client1.terminate()
client2.terminate()
client3.terminate()

server_log.close()
client1_log.close()
client2_log.close()
client3_log.close()

print("Server and clients stopped")

Server and clients stopped


---

## 6. Challenges and Solutions

### 6.1 Statistical Challenges

#### Problem: Non-IID Data

In real federated settings, data is **non-identically and independently distributed**:

- **Label distribution skew**: Client 1 has mostly class A, Client 2 has mostly class B
- **Feature distribution skew**: Different sensors, different environments
- **Quantity skew**: Some clients have 1000 samples, others have 10

**Impact:**
- Model bias toward clients with more data
- Slower convergence
- Lower accuracy

**Solutions:**

1. **Data augmentation**: Share small public dataset
2. **Personalization**: Per-client model layers
3. **Clustered FL**: Group similar clients
4. **Adaptive aggregation**: Weight updates differently

In [25]:
# Example: FedProx (handles non-IID better than FedAvg)
# Adds proximal term to objective

def fedprox_local_train(model, data, labels, global_weights, mu=0.01, epochs=5, lr=0.01):
    """
    FedProx local training with proximal term

    Args:
        model: Local model
        data: Training data
        labels: Training labels
        global_weights: Global model weights (for proximal term)
        mu: Proximal term coefficient
        epochs: Number of local epochs
        lr: Learning rate
    """
    dataset = TensorDataset(
        torch.FloatTensor(data),
        torch.LongTensor(labels)
    )
    dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr)

    model.train()
    for epoch in range(epochs):
        for batch_data, batch_labels in dataloader:
            optimizer.zero_grad()

            # Standard loss
            output = model(batch_data)
            loss = criterion(output, batch_labels)

            # Proximal term: prevents too much drift from global model
            proximal_term = 0
            for name, param in model.named_parameters():
                proximal_term += ((param - global_weights[name]) ** 2).sum()

            # Total loss with proximal regularization
            total_loss = loss + (mu / 2) * proximal_term

            total_loss.backward()
            optimizer.step()

    return model.state_dict()

print("FedProx implementation defined")
print("Key difference: Adds (mu/2)||w - w_global||^2 term to prevent drift")

FedProx implementation defined
Key difference: Adds (mu/2)||w - w_global||^2 term to prevent drift


### 6.2 Communication Challenges

#### Problem: Communication Bottleneck

- Mobile devices: Limited bandwidth, intermittent connectivity
- Large models: ResNet has ~25M parameters = ~100MB per round

**Solutions:**

1. **Gradient compression**: Send compressed updates
2. **Quantization**: Reduce precision (float32 → int8)
3. **Sparse updates**: Send only changed parameters
4. **Local iterations**: More computation, less communication

In [26]:
# Gradient compression example

def compress_gradient(gradient, compression_ratio=0.01):
    """
    Top-k compression: keep only largest k% gradients

    Args:
        gradient: Tensor to compress
        compression_ratio: Fraction of values to keep

    Returns:
        Compressed gradient with same shape
    """
    flat_grad = gradient.flatten()
    k = int(len(flat_grad) * compression_ratio)

    # Get top k indices by absolute value
    _, indices = torch.topk(torch.abs(flat_grad), k)

    # Create sparse gradient
    compressed = torch.zeros_like(flat_grad)
    compressed[indices] = flat_grad[indices]

    return compressed.reshape(gradient.shape)

# Test compression
test_gradient = torch.randn(1000)
compressed = compress_gradient(test_gradient, compression_ratio=0.1)

print(f"Original gradient size: {test_gradient.numel()} values")
print(f"Non-zero values after compression: {(compressed != 0).sum().item()}")
print(f"Compression ratio: {(compressed != 0).sum().item() / test_gradient.numel():.2%}")

Original gradient size: 1000 values
Non-zero values after compression: 100
Compression ratio: 10.00%


### 6.3 Privacy Challenges

#### Problem: Information Leakage

Even without sharing raw data, model updates can leak information:

- **Gradient inversion attacks**: Reconstruct training data from gradients
- **Membership inference**: Determine if specific sample was in training set
- **Model inversion**: Extract features about training data

**Solutions:**

1. **Differential Privacy**: Add noise to updates
2. **Secure Aggregation**: Encrypt updates, server only sees aggregate
3. **Homomorphic Encryption**: Compute on encrypted data

In [27]:
# Differential Privacy example

def add_gaussian_noise(gradient, epsilon=1.0, delta=1e-5, sensitivity=1.0):
    """
    Add Gaussian noise for differential privacy

    Args:
        gradient: Gradient tensor
        epsilon: Privacy budget (smaller = more privacy)
        delta: Probability of privacy failure
        sensitivity: L2 sensitivity of the query

    Returns:
        Noisy gradient
    """
    sigma = np.sqrt(2 * np.log(1.25 / delta)) * sensitivity / epsilon
    noise = torch.normal(0, sigma, size=gradient.shape)
    return gradient + noise

# Test differential privacy
original = torch.tensor([1.0, 2.0, 3.0, 4.0, 5.0])
noisy = add_gaussian_noise(original, epsilon=1.0)

print(f"Original gradient: {original.numpy()}")
print(f"Noisy gradient:    {noisy.numpy()}")
print(f"\nNoise magnitude: {(noisy - original).abs().mean():.4f}")
print("\nSmaller epsilon = more noise = more privacy")

Original gradient: [1. 2. 3. 4. 5.]
Noisy gradient:    [-4.2648687 -6.625432   7.413838   2.1275787 11.464632 ]

Noise magnitude: 5.3282

Smaller epsilon = more noise = more privacy


### 6.4 System Challenges

- **Client selection bias**: Always selecting fast/reliable clients
- **Straggler problem**: Slow clients delay rounds
- **Byzantine failures**: Malicious clients send bad updates

**Solutions:**
- Asynchronous FL
- Timeout mechanisms
- Byzantine-robust aggregation (median, trimmed mean)

---

## 7. Real-World Applications

### 7.1 Healthcare

**Use case:** Multi-hospital disease prediction

**Example:** COVID-19 diagnosis from chest X-rays
- 20 hospitals collaborate
- Each keeps patient data locally
- Shared model outperforms individual hospital models

**Benefits:**
- HIPAA compliance
- Larger effective dataset
- Better generalization

### 7.2 Financial Fraud Detection

**Use case:** Credit card fraud detection across banks

- Banks can't share transaction data (proprietary + regulatory)
- FL enables collaborative model training
- Better fraud detection than single-bank models

### 7.3 Smart Home Devices

- Voice assistants (wake word detection)
- Thermostats (energy optimization)
- Security cameras (object detection)

**Common benefit:** Learn from all devices without uploading sensitive home data

---

## 8. Hands-On Exercises

### Exercise 1: Understanding FedAvg

**Task:** Manually compute one round of FedAvg

Given:
- 3 clients with data sizes: n₁=100, n₂=150, n₃=50
- Initial weight: w₀ = 5.0
- After local training:
  - Client 1: w₁ = 4.5
  - Client 2: w₂ = 4.8
  - Client 3: w₃ = 4.2

**Question:** What is the global weight after aggregation?

In [28]:
# Exercise 1 Solution
n = [100, 150, 50]
w = [4.5, 4.8, 4.2]

total_n = sum(n)
w_global = sum([n[i] * w[i] for i in range(len(n))]) / total_n

print(f"Total samples: {total_n}")
print(f"Global weight after aggregation: {w_global:.2f}")

Total samples: 300
Global weight after aggregation: 4.60


### Exercise 2: Implement Non-IID Data Split

**Task:** Create a non-IID data split for MNIST where each client has only 2 classes

In [29]:
# Exercise 2: Non-IID MNIST split

def create_non_iid_mnist_simulation(num_clients=5):
    """
    Create non-IID data splits where each client has only 2 classes
    Using synthetic data for demonstration
    """
    # Simulate MNIST-like data (10 classes)
    import numpy as np
    np.random.seed(42)

    client_data = []
    classes_per_client = 2
    samples_per_class = 100
    #continue
    for i in range(num_clients):
            client_classes = [(i * 2) % 10, (i * 2 + 1) % 10]

            client_features = []
            client_labels = []

            for c in client_classes:
                X = np.random.randn(samples_per_class, 784) + c
                y = np.full(samples_per_class, c)

                client_features.append(X)
                client_labels.append(y)

            X_combined = np.vstack(client_features)
            y_combined = np.concatenate(client_labels)

            indices = np.random.permutation(len(y_combined))
            client_data.append((X_combined[indices], y_combined[indices]))

            print(f"Client {i} assigned classes: {client_classes}")


    return client_data

# Test the function
print("Creating non-IID data distribution:\n")
non_iid_data = create_non_iid_mnist_simulation(num_clients=5)
print("\nNote: Each client only has 2 out of 10 classes - this is non-IID!")

Creating non-IID data distribution:

Client 0 assigned classes: [0, 1]
Client 1 assigned classes: [2, 3]
Client 2 assigned classes: [4, 5]
Client 3 assigned classes: [6, 7]
Client 4 assigned classes: [8, 9]

Note: Each client only has 2 out of 10 classes - this is non-IID!


### Exercise 3: Communication Cost Analysis

**Task:** Calculate communication cost reduction

Scenario:
- 1000 clients
- Model size: 10 MB
- Traditional ML: Each client uploads all data
- FL: Each client uploads model updates

**Questions:**
1. If each client has 100MB of data, what's the communication cost for traditional ML?
2. If FL runs for 100 rounds with 10% client participation per round, what's the cost?
3. What's the reduction factor?

In [30]:
# Exercise 4 Solution

# Given parameters
num_clients = 1000
data_per_client_mb = 100
model_size_mb = 10
num_rounds = 100
participation_rate = 0.10

# Traditional ML: Upload all data
traditional_cost = num_clients * data_per_client_mb

print("Traditional Centralized ML:")
print(f"  Each of {num_clients} clients uploads {data_per_client_mb} MB")
print(f"  Total communication: {traditional_cost:,} MB = {traditional_cost/1000:.1f} GB\n")

# Federated Learning: Upload model updates
clients_per_round = int(num_clients * participation_rate)
# Each round: selected clients download model + upload updates
fl_cost_per_round = clients_per_round * (model_size_mb * 2)
fl_total_cost = num_rounds * fl_cost_per_round

print("Federated Learning:")
print(f"  {num_rounds} rounds × {clients_per_round} clients/round")
print(f"  Each client: download {model_size_mb} MB + upload {model_size_mb} MB = {model_size_mb*2} MB")
print(f"  Per round: {fl_cost_per_round:,} MB")
print(f"  Total communication: {fl_total_cost:,} MB = {fl_total_cost/1000:.1f} GB\n")

# Reduction
reduction_factor = traditional_cost / fl_total_cost
savings_percent = (1 - fl_total_cost / traditional_cost) * 100

print("Result:")
print(f"  Reduction factor: {reduction_factor:.1f}x")
print(f"  Communication savings: {savings_percent:.1f}%")
print(f"\n✓ Federated Learning uses {reduction_factor:.1f}x LESS communication!")

Traditional Centralized ML:
  Each of 1000 clients uploads 100 MB
  Total communication: 100,000 MB = 100.0 GB

Federated Learning:
  100 rounds × 100 clients/round
  Each client: download 10 MB + upload 10 MB = 20 MB
  Per round: 2,000 MB
  Total communication: 200,000 MB = 200.0 GB

Result:
  Reduction factor: 0.5x
  Communication savings: -100.0%

✓ Federated Learning uses 0.5x LESS communication!




## 9. Further Reading

### Foundational Papers

1. **McMahan et al. (2017)** - "Communication-Efficient Learning of Deep Networks from Decentralized Data"
   - Original FedAvg paper
   - https://arxiv.org/abs/1602.05629

2. **Kairouz et al. (2021)** - "Advances and Open Problems in Federated Learning"
   - Comprehensive survey (400+ pages)
   - https://arxiv.org/abs/1912.04977

3. **Li et al. (2020)** - "Federated Learning: Challenges, Methods, and Future Directions"
   - Great overview of challenges
   - https://arxiv.org/abs/1908.07873

### Key Algorithms

- **FedProx**: Handles heterogeneous data better
- **FedNova**: Normalized averaging for different local steps
- **Scaffold**: Variance reduction with control variates
- **FedBN**: Batch normalization for non-IID data

### Privacy-Preserving Techniques

- **Differential Privacy in FL**: Abadi et al. (2016)
- **Secure Aggregation**: Bonawitz et al. (2017)
- **Homomorphic Encryption**: Gentry (2009)

### Frameworks & Tools

| Framework | Language | Features |
|-----------|----------|----------|
| [Flower](https://flower.dev/) | Python | Easy to use, production-ready |
| [TensorFlow Federated](https://www.tensorflow.org/federated) | Python | Google's FL framework |
| [PySyft](https://github.com/OpenMined/PySyft) | Python | Privacy + FL |
| [FATE](https://fate.fedai.org/) | Python | Industrial FL platform |
| [FedML](https://fedml.ai/) | Python | Research + production |

### Online Courses

- **Federated Learning (TensorFlow)**: https://www.tensorflow.org/federated/tutorials/tutorials_overview
- **Privacy-Preserving ML (Coursera)**: Various courses on differential privacy

### Communities

- **OpenMined**: Privacy-preserving AI community
- **Flower Discuss**: Active FL forum
- **MachineLearning**: Subreddit with FL discussions

---

## Summary Checklist

After completing this notebook, you should understand:

- ✓ What federated learning is and why it matters
- ✓ The difference between centralized and federated learning
- ✓ How FedAvg algorithm works
- ✓ Main challenges: non-IID data, communication, privacy
- ✓ Basic implementation in PyTorch
- ✓ Real-world applications
- ✓ Privacy-preserving techniques (DP, secure aggregation)
- ✓ Where to learn more

---

## Quick Reference Card

```
┌─────────────────────────────────────────────────────────────┐
│           FEDERATED LEARNING QUICK REFERENCE                │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│ KEY EQUATION (FedAvg):                                      │
│   w_{t+1} = Σ(k=1 to K) (n_k/n) × w_k^{t+1}               │
│                                                             │
│ WORKFLOW:                                                   │
│   1. Server sends model to clients                          │
│   2. Clients train locally                                  │
│   3. Clients send updates to server                         │
│   4. Server aggregates updates                              │
│   5. Repeat                                                 │
│                                                             │
│ KEY PARAMETERS:                                             │
│   • C: Client fraction (0.1 = 10% participate)             │
│   • E: Local epochs (1-10 typical)                         │
│   • B: Batch size                                          │
│   • η: Learning rate                                        │
│                                                             │
│ CHALLENGES:                                                 │
│   • Non-IID data → Use FedProx, personalization            │
│   • Communication → Compress, quantize, local iterations    │
│   • Privacy → Differential privacy, secure aggregation     │
│   • Systems → Handle stragglers, Byzantine clients         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

